# 01 — Macro Analysis

Analyse globale du marché Steam pour identifier les tendances macro : publishers dominants, évolution temporelle, distribution des prix, et profil linguistique.

**Objectif métier** : Comprendre le paysage concurrentiel avant le lancement d'un nouveau jeu.

In [0]:
from pyspark.sql import functions as F

df = spark.table("steam_clean")
total_games = df.count()
print(f"Total games: {total_games}")
print(f"Distinct publishers: {df.filter(F.col('publisher').isNotNull()).select('publisher').distinct().count()}")

Total games: 55690
Distinct publishers: 29964


## Q1 — Which publisher has the most games?

Understanding market concentration helps position a new title against dominant publishers.

In [0]:
df_publishers = (df
    .filter(F.col("publisher").isNotNull())
    .groupBy("publisher")
    .agg(F.count("*").alias("game_count"))
    .orderBy(F.desc("game_count"))
    .limit(10)
)
display(df_publishers)

publisher,game_count
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
HH-Games,132
Sekai Project,132
Ubisoft,127
Laush Studio,126


Databricks visualization. Run in Databricks to view.

## Q2 — Which games have the best ratings?

A naive sort by `rating_ratio` favors niche games with few but unanimously positive reviews.
To surface genuinely well-rated games, we apply a **Bayesian average** (also known as IMDB weighted rating):

$$score = \frac{v \cdot R + m \cdot C}{v + m}$$

- $v$ = number of reviews for the game (`total_reviews`)
- $R$ = the game's own `rating_ratio`
- $m$ = minimum reviews required to be listed (threshold)
- $C$ = mean `rating_ratio` across games with >= 100000 reviews (prior)

This pulls low-sample games toward the global average, letting truly well-rated games with sufficient volume rise to the top.

In [0]:
m = 100000
C = df.filter(F.col("total_reviews") >= m).agg(
    F.avg("rating_ratio")
).first()[0]

df_best = (df
    .filter(F.col("total_reviews") >= m)
    .filter(F.col("name").isNotNull())
    .withColumn("bayesian_score",
        (F.col("total_reviews") * F.col("rating_ratio") + m * F.lit(C))
        / (F.col("total_reviews") + m)
    )
    .select(
        "name", "publisher", "rating_ratio",
        "total_reviews", "bayesian_score"
    )
    .orderBy(F.desc("bayesian_score"))
    .limit(20)
)
display(df_best)

name,publisher,rating_ratio,total_reviews,bayesian_score
Terraria,Re-Logic,0.978420408623737,1037091,0.969548773756837
Wallpaper Engine,Wallpaper Engine Team,0.9807193158162436,572127,0.965368426948978
Stardew Valley,ConcernedApe,0.9816845914201889,506841,0.9645231365381306
Left 4 Dead 2,Valve,0.9745286560187932,660664,0.9617783734999102
Portal 2,Valve,0.9878167405094994,309441,0.9608837041232698
Euro Truck Simulator 2,SCS Software,0.9734431097497717,587983,0.9595036282872335
Garry's Mod,Valve,0.9663412017889722,891238,0.9573827725530455
Phasmophobia,Kinetic Games,0.968350512524995,491098,0.9529878035451579
The Witcher 3: Wild Hunt,CD PROJEKT RED,0.9616262738040227,657872,0.9505314679786768
Hades,Supergiant Games,0.9860495391761881,202789,0.9502134644915625


Databricks visualization. Run in Databricks to view.

## Q3 — Which years had the most releases? Covid impact?

Tracking release volume year-over-year to detect market shifts around 2019-2021.

In [0]:
df_yearly = (df
    .filter(F.col("release_year").isNotNull())
    .groupBy("release_year")
    .agg(F.count("*").alias("releases"))
    .orderBy("release_year")
)
display(df_yearly)

release_year,releases
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61


Databricks visualization. Run in Databricks to view.

In [0]:
df_covid = df_yearly.filter(F.col("release_year").between(2018, 2022))
df_prev = df_yearly.select(
    (F.col("release_year") + 1).alias("release_year"),
    F.col("releases").alias("prev_year_releases")
)
df_yoy = (df_covid
    .join(df_prev, on="release_year", how="left")
    .withColumn("yoy_growth_pct",
        F.round((F.col("releases") - F.col("prev_year_releases"))
                / F.col("prev_year_releases") * 100, 1))
    .orderBy("release_year")
)
display(df_yoy)

release_year,releases,prev_year_releases,yoy_growth_pct
2018,7678,6017,27.6
2019,6968,7678,-9.2
2020,8305,6968,19.2
2021,8823,8305,6.2
2022,7455,8823,-15.5


Databricks visualization. Run in Databricks to view.

## Q4 — How are prices and discounts distributed?

Price positioning is a key go-to-market decision; understanding the distribution informs pricing strategy.

In [0]:
free_count = F.sum(F.col("is_free").cast("int"))
df_price_stats = df.select(
    F.round(F.min("price_usd"), 2).alias("min_price"),
    F.round(F.max("price_usd"), 2).alias("max_price"),
    F.round(F.avg("price_usd"), 2).alias("avg_price"),
    F.round(
        F.percentile_approx("price_usd", 0.5), 2
    ).alias("median_price"),
    free_count.alias("free_games"),
    F.round(
        free_count / F.count("*") * 100, 1
    ).alias("pct_free")
)
display(df_price_stats)

min_price,max_price,avg_price,median_price,free_games,pct_free
0.0,999.0,7.73,4.99,7779,14.0


In [0]:
has_discount = (F.col("discount") > 0).cast("int")
discount_sum = F.sum(has_discount)
df_discount = df.select(
    discount_sum.alias("games_with_discount"),
    F.round(
        discount_sum / F.count("*") * 100, 1
    ).alias("pct_discounted"),
    F.round(F.avg(
        F.when(
            F.col("discount") > 0,
            F.col("discount")
        )
    ), 1).alias("avg_discount_when_exists")
)
display(df_discount)

games_with_discount,pct_discounted,avg_discount_when_exists
2518,4.5,57.6


In [0]:
df_prices = (df
    .filter(F.col("price_usd") > 0)
    .filter(F.col("price_usd") <= 100)
    .withColumn("price_bracket",
        F.when(F.col("price_usd") <= 5, "01_0-5$")
        .when(F.col("price_usd") <= 10, "02_5-10$")
        .when(F.col("price_usd") <= 20, "03_10-20$")
        .when(F.col("price_usd") <= 30, "04_20-30$")
        .when(F.col("price_usd") <= 50, "05_30-50$")
        .otherwise("06_50-100$")
    )
    .groupBy("price_bracket")
    .agg(F.count("*").alias("count"))
    .orderBy("price_bracket")
)
display(df_prices)

price_bracket,count
01_0-5$,23498
02_5-10$,12440
03_10-20$,9017
04_20-30$,1790
05_30-50$,838
06_50-100$,291


Databricks visualization. Run in Databricks to view.

In [0]:
df_price_discount = (df
    .filter(F.col("price_usd") > 0)
    .filter(F.col("price_usd") <= 100)
    .withColumn("price_bracket",
        F.when(F.col("price_usd") <= 5, "01_0-5$")
        .when(F.col("price_usd") <= 10, "02_5-10$")
        .when(F.col("price_usd") <= 20, "03_10-20$")
        .when(F.col("price_usd") <= 30, "04_20-30$")
        .when(F.col("price_usd") <= 50, "05_30-50$")
        .otherwise("06_50-100$")
    )
    .withColumn("has_discount", (F.col("discount") > 0).cast("int"))
    .groupBy("price_bracket")
    .agg(
        F.count("*").alias("total_games"),
        F.sum("has_discount").alias("discounted_count"),
        F.round(F.sum("has_discount") / F.count("*") * 100, 1).alias("pct_with_discount"),
        F.round(F.avg(
            F.when(F.col("discount") > 0, F.col("discount"))
        ), 1).alias("avg_discount_when_exists"),
    )
    .orderBy("price_bracket")
)
display(df_price_discount)

price_bracket,total_games,discounted_count,pct_with_discount,avg_discount_when_exists
01_0-5$,23498,1884,8.0,63.6
02_5-10$,12440,380,3.1,45.0
03_10-20$,9017,211,2.3,31.6
04_20-30$,1790,31,1.7,33.2
05_30-50$,838,10,1.2,34.5
06_50-100$,291,1,0.3,25.0


Databricks visualization. Run in Databricks to view.

## Q5 — Which languages are most represented?

Language support drives accessibility and market reach; knowing the baseline helps prioritize localization.

In [0]:
df_languages = (df
    .filter(F.col("languages").isNotNull())
    .withColumn("language_single", F.explode(F.col("languages")))
    .groupBy("language_single")
    .agg(F.count("*").alias("game_count"))
    .orderBy(F.desc("game_count"))
    .limit(15)
)
display(df_languages)

language_single,game_count
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6599


Databricks visualization. Run in Databricks to view.

## Q6 — What proportion of games is age-restricted (16+/18+)?

Age restrictions affect target audience size and distribution channel eligibility.

In [0]:
df_age = (df
    .groupBy("required_age_clean")
    .agg(F.count("*").alias("count"))
    .orderBy("required_age_clean")
)
display(df_age)

required_age_clean,count
0,55029
3,3
5,1
6,4
7,3
8,3
9,1
10,7
12,32
13,26


Databricks visualization. Run in Databricks to view.

In [0]:
df_restricted = df.filter(F.col("required_age_clean") >= 16).select(
    F.count("*").alias("restricted_games"),
    F.round(F.count("*") / F.lit(total_games) * 100, 2).alias("pct_restricted")
)
display(df_restricted)

restricted_games,pct_restricted
306,0.55


## Conclusion — Macro Analysis

Sur les **55 690 jeux** du catalogue Steam, impliquant **29 964 publishers distincts**, voici les enseignements par question.

### Q1 — Publisher dominant

**Big Fish Games** domine avec **422 jeux**, suivi de 8floor (202) et SEGA (165). Ubisoft se classe 9e avec **127 jeux**. Le marche reste tres fragmente : le top 10 ne represente qu'une fraction marginale du catalogue, signe d'un marche domine par les editeurs independants.

### Q2 — Jeux les mieux notes (score bayesien, m=100 000)

Le classement utilise un score bayesien qui penalise les jeux a faible volume de reviews. Top 3 :

| Rang | Jeu | Publisher | Reviews | Score bayesien |
|------|-----|-----------|---------|----------------|
| 1 | Terraria | Re-Logic | 1037091 | 0.97 |
| 2 | Wallpaper Engine | Wallpaper Engine Team | 572127 | 0.97 |
| 3 | Stardew Valley | ConcernedApe | 506841 | 0.96 |

Les jeux les mieux notes combinent volume eleve et ratio positif quasi unanime (>96%). Valve place 3 titres dans le top 10 (Left 4 Dead 2, Portal 2, Garry's Mod).

### Q3 — Annees de sortie et impact Covid

Le pic de sorties est atteint en **2021 avec 8 823 jeux**. Evolution Year-over-Year autour du Covid :

| Annee | Sorties | YoY |
|-------|---------|-----|
| 2018 | 7 678 | +27.6% |
| 2019 | 6 968 | -9.2% |
| 2020 | 8 305 | +19.2% |
| 2021 | 8 823 | +6.2% |
| 2022 | 7 455 | -15.5% |

Le Covid a provoque un rebond de **+19.2% en 2020** apres une contraction en 2019, suivi d'un pic en 2021. La retombee de **-15.5% en 2022** suggere un retour a la normale post-pandemie.

### Q4 — Distribution des prix et discounts

**Prix** : moyenne a **7.73$**, mediane a **4.99$**, maximum a 999$. **7 779 jeux gratuits (14.0%)**. La tranche 0-5$ concentre **23 498 jeux** payants, soit la majorite du catalogue payant. Au-dela de 30$, on ne trouve que 1 129 jeux.

**Discounts** : seuls **2 518 jeux (4.5%)** affichent un discount au moment du snapshot, avec un discount moyen de **57.6%** quand il existe.

### Q5 — Langues les plus representees

L'anglais est quasi universel avec **55 116 jeux** (98.97%). Le top 5 :

| Langue | Jeux |
|--------|------|
| English | 55 116 |
| German | 14 019 |
| French | 13 426 |
| Russian | 12 922 |
| Simplified Chinese | 12 782 |

Les langues europeennes (allemand, francais) et le russe devancent le chinois simplifie. L'espagnol (12233) et le japonais (10 368) completent le top 7.

### Q6 — Jeux avec restriction d'age (16+/18+)

**306 jeux** sont restreints 16+/18+, soit **0.55%** du catalogue. Le detail : 38 jeux a 16 ans, 38 a 17 ans, **227 jeux a 18 ans** (74% des jeux restreints) et 3 jeux avec des seuils atypiques (20, 21, 35 ans — 1 jeu chacun). Le marche Steam est massivement tous publics : **55 029 jeux (98.8%)** n'ont aucune restriction d'age.

### Limites

- **Survivorship bias** : le dataset ne contient que les jeux encore presents sur le store. Les jeux retires (echecs commerciaux, violations de droits) sont absents, ce qui biaise les distributions de notes et de prix vers le haut.
- **Snapshot bias pour les discounts** : le taux de 4.5% de jeux en promotion ne reflete qu'un instant t. Pendant les Steam Sales, ce chiffre serait significativement plus eleve.
- **Publisher = label, pas entite juridique** : un meme groupe peut publier sous plusieurs noms (ex. Square Enix, Square Enix Europe), fragmentant artificiellement les comptages.
- **Absence de donnees de ventes** : sans volumes de ventes, l'analyse reste limitee a l'offre (catalogue) et non a la demande (revenus, popularite reelle).